In [ ]:
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import SimpleITK as sitk
import numpy as np

In [ ]:

cut_off_date = '2026-05-28'
mount_point = "/mnt/GWH/Groups/FAMLI/Shared/C1_ML_Analysis/famli_ml_lists/dataset_list/"

# out_dir = '/mnt/GWH/Groups/FAMLI/Shared/Ben/naami_data_analysis/img_translation/'
# version = '2026_'

# def build_naami_df(train_fn, valid_fn, test_fn):

#     csv_naami_visit = "/mnt/GWH/Groups/FAMLI/Shared/Ben/naami_data_analysis/visit_table_csv/naami_2026_feb_visit_table.csv"
#     csv_naami_instance = "/mnt/GWH/Groups/FAMLI/Shared/Ben/naami_data_analysis/instance_table_csv/naami_2026_feb_research_scans_instance_table.csv"

#     df_naami_visit = pd.read_csv(csv_naami_visit)
#     df_naami_instance = pd.read_csv(csv_naami_instance)

#     df_naami = df_naami_instance.merge(df_naami_visit, on='study_id', how='left')

#     df_naami['file_path'] = df_naami['file_path'].apply(lambda x: os.path.join('naami_masked_resampled_256_spc075', x.replace('.dcm', '.nrrd')))

#     df_naami_train, df_naami_valid, df_naami_test = df_naami[df_naami['split'] == 'train'], df_naami[df_naami['split'] == 'tune'], df_naami[df_naami['split'] == 'test']
    

#     df_naami_train.to_csv(train_fn, index=False)
#     df_naami_valid.to_csv(valid_fn, index=False)
#     df_naami_test.to_csv(test_fn, index=False)

#     return df_naami_train, df_naami_valid, df_naami_test


# train_fn = os.path.join(out_dir, f'{version}_naami_leletk_train.csv')
# valid_fn = os.path.join(out_dir, f'{version}_naami_leletk_valid.csv')
# test_fn = os.path.join(out_dir, f'{version}_naami_leletk_test.csv')

# if not os.path.exists(train_fn):
#     df_train_leltek, df_valid_leltek, df_test_leltek = build_naami_df(train_fn, valid_fn, test_fn)
# else:
#     df_train_leltek = pd.read_csv(train_fn)
#     df_valid_leltek = pd.read_csv(valid_fn)
#     df_test_leltek = pd.read_csv(test_fn)

df = pd.read_csv(os.path.join(mount_point, cut_off_date, "c3_instance_blindsweep_analysis_dataset.csv"))
df_leltek = df.query('model == "LK128"')

df_leltek['file_path'] = df_leltek['file_path'].apply(lambda x: x.replace('Groups/FAMLI/Restricted_access_data/Ultrasound/Dataset_C3/', 'Dataset_C3_masked_resampled_256_spc075/').replace('.dcm', '.nrrd'))

df_train_leltek = df_leltek.query('split == "train"')
df_valid_leltek = df_leltek.query('split == "tune"')
df_test_leltek = df_leltek.query('split == "test"')


In [ ]:
df_leltek['file_path'].head().iloc[0]

In [ ]:
csv_train_all = f"{mount_point}/{cut_off_date}/img_translation_all_manufacturers_train.csv"
csv_train_iq3 = f"{mount_point}/{cut_off_date}/img_translation_iq3_train.csv"

csv_valid_all = f"{mount_point}/{cut_off_date}/img_translation_all_manufacturers_tune.csv"
csv_valid_iq3 = f"{mount_point}/{cut_off_date}/img_translation_iq3_tune.csv"

df_train_all = pd.read_csv(csv_train_all)
df_train_iq3 = pd.read_csv(csv_train_iq3)
df_valid_all = pd.read_csv(csv_valid_all)
df_valid_iq3 = pd.read_csv(csv_valid_iq3)


# df_train_leltek['manufacturer'] = 'LELTEK'
# df_train_leltek['model'] = 'LK128C'

# df_valid_leltek['manufacturer'] = 'LELTEK'
# df_valid_leltek['model'] = 'LK128C'

df_train = pd.concat([df_train_all, df_train_iq3, df_train_leltek])
df_valid = pd.concat([df_valid_all, df_valid_iq3, df_valid_leltek])


keep_manufacturers = ['GE Healthcare', 'Sonosite, Inc.', 'Clarius Mobile', 'Butterfly Network Inc', "UNC Global Women's Health"]
df_train = df_train[df_train['manufacturer'].isin(keep_manufacturers)]
df_valid = df_valid[df_valid['manufacturer'].isin(keep_manufacturers)]



In [ ]:
df_train[['manufacturer', 'model']].value_counts()

In [ ]:
df_valid[['manufacturer', 'model']].value_counts()

In [ ]:
# Randomly sample min count per manufacturer

min_count_train = df_train['manufacturer'].value_counts().min()

df_train_balanced = df_train.groupby('manufacturer').apply(lambda x: x.sample(min(x.shape[0], min_count_train), random_state=42))

df_train_balanced['manufacturer'].value_counts()



In [ ]:
min_count_valid = df_valid['manufacturer'].value_counts().min()
df_valid_balanced = df_valid.groupby('manufacturer').apply(lambda x: x.sample(min(x.shape[0], min_count_valid), random_state=42))
df_valid_balanced['manufacturer'].value_counts()

In [ ]:
out_dir = os.path.join(mount_point, cut_off_date)

def build_ds(out_dir, df_train, df_valid):

    # build train and valid datasets per manufacturer ALL v.s. 1

    manufacturers = df_train['manufacturer'].unique()

    file_paths = []

    for manufacturer in manufacturers:

        df_train_all = df_train[df_train['manufacturer'] != manufacturer]
        df_train_manufacturer = df_train[df_train['manufacturer'] == manufacturer]

        df_valid_all = df_valid[df_valid['manufacturer'] != manufacturer]
        df_valid_manufacturer = df_valid[df_valid['manufacturer'] == manufacturer]

        out_name_manufacturer = manufacturer.split(" ")[0].replace(",", "").replace(".", "")

        # df_train_manufacturer = df_train_manufacturer[df_train_manufacturer['file_path'].apply(lambda x: os.path.exists(os.path.join(mount_point, x)))]
        # df_valid_manufacturer = df_valid_manufacturer[df_valid_manufacturer['file_path'].apply(lambda x: os.path.exists(os.path.join(mount_point, x)))]
        df_train_manufacturer.to_csv(os.path.join(out_dir, f'ALL_vs_{out_name_manufacturer}_train_manufacturer_balanced.csv'), index=False)
        df_valid_manufacturer.to_csv(os.path.join(out_dir, f'ALL_vs_{out_name_manufacturer}_valid_manufacturer_balanced.csv'), index=False)

        # df_train_all = df_train_all[df_train_all['file_path'].apply(lambda x: os.path.exists(os.path.join(mount_point, x)))]
        # df_valid_all = df_valid_all[df_valid_all['file_path'].apply(lambda x: os.path.exists(os.path.join(mount_point, x)))]

        df_train_all.to_csv(os.path.join(out_dir, f'ALL_vs_{out_name_manufacturer}_train_all_balanced.csv'), index=False)
        df_valid_all.to_csv(os.path.join(out_dir, f'ALL_vs_{out_name_manufacturer}_valid_all_balanced.csv'), index=False)

        print(f"Manufacturer: {manufacturer}")
        print("TRAIN MANUFACTURER:", df_train_manufacturer['manufacturer'].value_counts())
        print("VALID MANUFACTURER:", df_valid_manufacturer['manufacturer'].value_counts())
        print("TRAIN ALL:", df_train_all['manufacturer'].value_counts())
        print("VALID ALL:", df_valid_all['manufacturer'].value_counts())
        print("--------------------------------")

        file_paths.append(pd.concat([df_train_manufacturer['file_path'], df_valid_manufacturer['file_path'], df_train_all['file_path'], df_valid_all['file_path']]))

    file_paths = pd.concat(file_paths).drop_duplicates().reset_index(drop=True)
    file_paths.to_csv(os.path.join(out_dir, 'file_paths.csv'), index=False)

# build_ds(out_dir, df_train_balanced, df_valid_balanced)

